In [2]:
import pandas as pd

In [3]:
order_products_prior = pd.read_csv('data/order_products__prior.csv') 
order_products_train = pd.read_csv('data/order_products__train.csv') 

print(order_products_prior.shape)
print(order_products_prior.head())

print(order_products_train.shape)
print(order_products_train.head())

(32434489, 4)
   order_id  product_id  add_to_cart_order  reordered
0         2       33120                  1          1
1         2       28985                  2          1
2         2        9327                  3          0
3         2       45918                  4          1
4         2       30035                  5          0
(1384617, 4)
   order_id  product_id  add_to_cart_order  reordered
0         1       49302                  1          1
1         1       11109                  2          1
2         1       10246                  3          0
3         1       49683                  4          0
4         1       43633                  5          1


In [4]:
order_products_prior[order_products_prior['order_id'] == 2]

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0
5,2,17794,6,1
6,2,40141,7,1
7,2,1819,8,1
8,2,43668,9,0


In [18]:
products = pd.read_csv("data/products.csv")

In [19]:
order_products_train_new = order_products_train.merge(products, on="product_id", how="left")

In [20]:
order_products_train_new.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,1,49302,1,1,Bulgarian Yogurt,120,16
1,1,11109,2,1,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16
2,1,10246,3,0,Organic Celery Hearts,83,4
3,1,49683,4,0,Cucumber Kirby,83,4
4,1,43633,5,1,Lightly Smoked Sardines in Olive Oil,95,15


In [21]:
#transactions = order_products_train.groupby('order_id')['product_id'].apply(list).tolist()

transactions = order_products_train_new.groupby('order_id')['product_name'].apply(list).tolist()


In [22]:
from mlxtend.frequent_patterns import apriori, fpgrowth

In [23]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

df = pd.DataFrame(te_ary, columns=te.columns_)

In [24]:
print(df.shape)

(131209, 39123)


In [25]:
from mlxtend.frequent_patterns import apriori, fpgrowth


# FP-Growth
fp_result = fpgrowth(
    df,
    min_support=0.001,
    use_colnames=True
)

In [26]:
fp_result[fp_result['itemsets'].apply(len) > 3]


,support,itemsets
1873,0.001242,"frozenset({Bag of Organic Bananas, Organic Str..."
2083,0.001738,"frozenset({Organic Strawberries, Bag of Organi..."
2153,0.001067,"frozenset({Bag of Organic Bananas, Organic Str..."
2753,0.001113,"frozenset({Limes, Banana, Organic Avocado, Lar..."
3317,0.001021,"frozenset({Organic Avocado, Banana, Organic Ba..."


In [27]:
print(len(fp_result))

4444


In [28]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(fp_result, metric="lift", min_threshold=1.0)
strong = rules[(rules["lift"] > 1.5) & (rules["confidence"] > 0.1)]
print(len(strong))
print(strong.sort_values("lift", ascending=False).head(20)[
    ["antecedents", "consequents", "support", "confidence", "lift"]])

2324
                                            antecedents  \
6644            frozenset({Strawberry Rhubarb Yoghurt})   
6645                     frozenset({Blueberry Yoghurt})   
6692  frozenset({Nonfat Icelandic Style Strawberry Y...   
6693  frozenset({Icelandic Style Skyr Blueberry Non-...   
6739  frozenset({Icelandic Style Skyr Blueberry Non-...   
6738   frozenset({Non Fat Acai & Mixed Berries Yogurt})   
6780           frozenset({Kiwi Sandia Sparkling Water})   
6781   frozenset({Blackberry Cucumber Sparkling Water})   
6762              frozenset({Non Fat Raspberry Yogurt})   
6763  frozenset({Icelandic Style Skyr Blueberry Non-...   
4154            frozenset({Grapefruit Sparkling Water})   
4155                 frozenset({Lemon Sparkling Water})   
6773  frozenset({Total 2% Lowfat Greek Strained Yogu...   
6772  frozenset({Total 2% Lowfat Greek Strained Yogu...   
4833  frozenset({Total 2% Greek Strained Yogurt with...   
4832  frozenset({Total 2% Lowfat Greek Strained Yog

In [29]:
print("Lift ≥ 1.5:", (rules["lift"] >= 1.5).sum())
print("Lift ≥ 1.5 & confidence ≥ 0.1:",
      ((rules["lift"] >= 1.5) & (rules["confidence"] >= 0.1)).sum())

Lift ≥ 1.5: 6178
Lift ≥ 1.5 & confidence ≥ 0.1: 2324


In [30]:
for lift_t, conf_t in [(2, 0.15), (3, 0.2), (5, 0.25)]:
    n = ((rules["lift"] >= lift_t) & (rules["confidence"] >= conf_t)).sum()
    print(f"lift ≥ {lift_t}, conf ≥ {conf_t}: {n}")

lift ≥ 2, conf ≥ 0.15: 1334
lift ≥ 3, conf ≥ 0.2: 615
lift ≥ 5, conf ≥ 0.25: 125


In [33]:
final = rules[(rules["lift"] >= 3) & (rules["confidence"] >= 0.25)].copy()
final["pair"] = final.apply(lambda r: frozenset(r["antecedents"] | r["consequents"]), axis=1)
final = final.sort_values("confidence", ascending=False).drop_duplicates("pair")

print(len(final))
final.sort_values("lift", ascending=False).head(96)[
    ["antecedents", "consequents", "support", "confidence", "lift"]]

304


,antecedents,consequents,support,confidence,lift
6645,frozenset({Blueberry Yoghurt}),frozenset({Strawberry Rhubarb Yoghurt}),0.001197,0.310277,80.298014
6692,frozenset({Nonfat Icelandic Style Strawberry Y...,frozenset({Icelandic Style Skyr Blueberry Non-...,0.001166,0.422652,78.660621
6738,frozenset({Non Fat Acai & Mixed Berries Yogurt}),frozenset({Icelandic Style Skyr Blueberry Non-...,0.001288,0.402381,74.887947
6780,frozenset({Kiwi Sandia Sparkling Water}),frozenset({Blackberry Cucumber Sparkling Water}),0.001014,0.286022,72.449026
6762,frozenset({Non Fat Raspberry Yogurt}),frozenset({Icelandic Style Skyr Blueberry Non-...,0.001677,0.381944,71.084466
...,...,...,...,...,...
2800,"frozenset({Limes, Banana})",frozenset({Organic Avocado}),0.002683,0.264463,4.683480
6496,"frozenset({Organic Hass Avocado, Organic Kiwi})",frozenset({Organic Strawberries}),0.001029,0.387931,4.672301
1361,"frozenset({Organic Yellow Onion, Organic Straw...",frozenset({Organic Hass Avocado}),0.001326,0.259314,4.665349
1022,"frozenset({Organic Grape Tomatoes, Organic Ras...",frozenset({Organic Strawberries}),0.001136,0.387013,4.661244


In [34]:
# write final rules to a CSV file
final.to_csv("final_rules.csv", index=False)